<a href="https://colab.research.google.com/github/akanksha1723/AI_Projects/blob/main/Day20.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### **RAG-Based Question Answering Assistant**

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import time # For simulating LLM response time

# Re-using the get_embedding and retrieve_similar_memories functions from before
# 1. Simulate an Embedding Model
def get_embedding(text):
    seed = sum(ord(c) for c in text) % 1000 # Keep consistent dummy embeddings
    np.random.seed(seed)
    return np.random.rand(10)

# 2. Prepare our initial 'memory' documents
memory_documents = [
    "The user is interested in simple chatbots.",
    "The chatbot's name is Chatbot.",
    "The conversation included a joke about atoms.",
    "The user asked how to save conversation to a file.",
    "The user wants to append new entries to the conversation file.",
    "The current topic is upgrading to agentic AI."
]

# 3. Create our initial 'Vector Database'
vector_db = []
for doc in memory_documents:
    embedding = get_embedding(doc)
    vector_db.append({'text': doc, 'embedding': embedding})

print(f"Initialized in-memory vector store with {len(vector_db)} entries.")

# 4. Implement a Retrieval Function
def retrieve_similar_memories(query_text, top_k=2):
    query_embedding = get_embedding(query_text)
    similarities = []

    for entry in vector_db:
        doc_embedding = entry['embedding'].reshape(1, -1)
        query_embedding_reshaped = query_embedding.reshape(1, -1)

        similarity = cosine_similarity(query_embedding_reshaped, doc_embedding)[0][0]
        similarities.append((entry['text'], similarity))

    similarities.sort(key=lambda x: x[1], reverse=True)
    return similarities[:top_k]

# 5. Simulate an LLM Response Generation (simplified for demonstration)
def generate_llm_response(query, retrieved_context):
    print("\n[Simulating LLM thinking...]", end='')
    time.sleep(1) # Simulate processing time

    # In a real scenario, this would be a call to an LLM API
    # The LLM would synthesize information from 'query' and 'retrieved_context'
    if retrieved_context:
        context_str = ", ".join([mem[0] for mem in retrieved_context])
        response = f"Based on our past discussions (e.g., '{context_str}'), regarding your query about '{query}', I'd say... [LLM generates detailed response]."
    else:
        response = f"Regarding your query about '{query}', I'd say... [LLM generates a general response]."

    # Add a bit of dynamic response for demonstration
    if "chatbot" in query.lower() and not retrieved_context:
        response = "Ah, you're asking about chatbots! I remember we discussed simple rule-based ones earlier."
    elif "memory" in query.lower() and retrieved_context and any("agentic AI" in m[0].lower() for m in retrieved_context):
        response = "Yes, we were just talking about agentic AI and how vector databases like this one help provide continuous memory!"
    elif "file" in query.lower() and retrieved_context and any("save conversation" in m[0].lower() for m in retrieved_context):
        response = "Right, we covered how to save and append chat conversations to a text file using Python's 'w' and 'a' modes."

    return response


# 6. Implement the Continuous Memory Loop
print("\n--- Starting Continuous Memory Loop (Type 'quit' to exit) ---")
while True:
    user_input = input("\nYou: ")
    if user_input.lower() == 'quit':
        print("Agent: Goodbye! Memory loop terminated.")
        break

    # Step 1: User Input (already have user_input)

    # Step 2: Memory Retrieval
    retrieved_memories = retrieve_similar_memories(user_input)
    print(f"[Agent retrieved: {len(retrieved_memories)} relevant memories]")
    for text, score in retrieved_memories:
        print(f" - '{text}' (Similarity: {score:.2f})")

    # Step 3 & 4: Context Augmentation and Response Generation
    agent_response = generate_llm_response(user_input, retrieved_memories)
    print("\nAgent: " + agent_response)

    # Step 5: Memory Update
    # Add the user's input to memory
    user_memory_entry = f"User said: {user_input}"
    user_embedding = get_embedding(user_memory_entry)
    vector_db.append({'text': user_memory_entry, 'embedding': user_embedding})
    memory_documents.append(user_memory_entry) # Keep memory_documents in sync

    # Add the agent's response to memory
    agent_memory_entry = f"Agent responded: {agent_response}"
    agent_embedding = get_embedding(agent_memory_entry)
    vector_db.append({'text': agent_memory_entry, 'embedding': agent_embedding})
    memory_documents.append(agent_memory_entry)

    print(f"[Memory updated. Total entries in vector_db: {len(vector_db)}]")

Initialized in-memory vector store with 6 entries.

--- Starting Continuous Memory Loop (Type 'quit' to exit) ---

You: chatbot
[Agent retrieved: 2 relevant memories]
 - 'The user wants to append new entries to the conversation file.' (Similarity: 0.90)
 - 'The chatbot's name is Chatbot.' (Similarity: 0.84)

[Simulating LLM thinking...]
Agent: Based on our past discussions (e.g., 'The user wants to append new entries to the conversation file., The chatbot's name is Chatbot.'), regarding your query about 'chatbot', I'd say... [LLM generates detailed response].
[Memory updated. Total entries in vector_db: 8]

You: memory
[Agent retrieved: 2 relevant memories]
 - 'The chatbot's name is Chatbot.' (Similarity: 0.89)
 - 'The user asked how to save conversation to a file.' (Similarity: 0.82)

[Simulating LLM thinking...]
Agent: Based on our past discussions (e.g., 'The chatbot's name is Chatbot., The user asked how to save conversation to a file.'), regarding your query about 'memory', I'd sa